In [1]:
PythonFinalizationError

PythonFinalizationError

In [2]:
import sys
from pathlib import Path

import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

PROJECT_ROOT = Path.cwd()

if not (PROJECT_ROOT / "src").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT / "src"))

from xray_classifier.models.cnn import XRayCNN


DATASET_ROOT = Path("/Users/afaquemogni/Downloads/deep_learning/Data")

device = torch.device(
    "mps"
    if torch.backends.mps.is_available()
    else "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print(f"Project root: {PROJECT_ROOT}")
print(f"Device: {device}")
print(f"Dataset exists: {DATASET_ROOT.exists()}")

Project root: /Users/afaquemogni/Downloads/deep_learning/xray_lung_classifier
Device: mps
Dataset exists: True


In [3]:
IMAGE_SIZE = 224
BATCH_SIZE = 32

train_transform = transforms.Compose(
    [
        transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.5, 0.5, 0.5],
            std=[0.5, 0.5, 0.5],
        ),
    ]
)

evaluation_transform = transforms.Compose(
    [
        transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.5, 0.5, 0.5],
            std=[0.5, 0.5, 0.5],
        ),
    ]
)

In [4]:
train_dataset = datasets.ImageFolder(
    DATASET_ROOT / "train",
    transform=train_transform,
)

test_dataset = datasets.ImageFolder(
    DATASET_ROOT / "test",
    transform=evaluation_transform,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
)

print(f"Class names: {train_dataset.classes}")
print(f"Class mapping: {train_dataset.class_to_idx}")
print(f"Training images: {len(train_dataset)}")
print(f"Test images: {len(test_dataset)}")

Class names: ['COVID19', 'NORMAL', 'PNEUMONIA']
Class mapping: {'COVID19': 0, 'NORMAL': 1, 'PNEUMONIA': 2}
Training images: 5144
Test images: 1288


In [5]:
images, labels = next(iter(train_loader))

print(f"Image batch shape: {images.shape}")
print(f"Label batch shape: {labels.shape}")
print(f"First 10 labels: {labels[:10].tolist()}")

Image batch shape: torch.Size([32, 3, 224, 224])
Label batch shape: torch.Size([32])
First 10 labels: [2, 1, 2, 2, 2, 2, 2, 2, 2, 1]


In [6]:
class_names = train_dataset.classes

model = XRayCNN(num_classes=len(class_names)).to(device)

class_counts = torch.bincount(torch.tensor(train_dataset.targets)).float()

class_weights = class_counts.sum() / (
    len(class_counts) * class_counts
)

loss_function = torch.nn.CrossEntropyLoss(
    weight=class_weights.to(device)
)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001,
)

EPOCHS = 5

train_losses = []
train_accuracies = []

print(f"Class counts: {class_counts.tolist()}")
print(f"Class weights: {class_weights.tolist()}")
print(f"Model classes: {class_names}")

Class counts: [460.0, 1266.0, 3418.0]
Class weights: [3.727536201477051, 1.3543970584869385, 0.5016579031944275]
Model classes: ['COVID19', 'NORMAL', 'PNEUMONIA']


In [7]:
def train_one_epoch(model, loader, optimizer, loss_function, device):
    model.train()

    total_loss = 0.0
    correct_predictions = 0
    total_images = 0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        logits = model(images)
        loss = loss_function(logits, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)

        predictions = logits.argmax(dim=1)
        correct_predictions += (predictions == labels).sum().item()
        total_images += labels.size(0)

    average_loss = total_loss / total_images
    accuracy = correct_predictions / total_images

    return average_loss, accuracy

In [8]:
for epoch in range(1, EPOCHS + 1):
    loss, accuracy = train_one_epoch(
        model,
        train_loader,
        optimizer,
        loss_function,
        device,
    )

    train_losses.append(loss)
    train_accuracies.append(accuracy)

    print(
        f"Epoch {epoch}/{EPOCHS} | "
        f"loss: {loss:.4f} | "
        f"training accuracy: {accuracy:.2%}"
    )

Epoch 1/5 | loss: 0.5430 | training accuracy: 76.94%
Epoch 2/5 | loss: 0.3298 | training accuracy: 85.32%
Epoch 3/5 | loss: 0.2945 | training accuracy: 86.82%
Epoch 4/5 | loss: 0.2493 | training accuracy: 89.33%
Epoch 5/5 | loss: 0.2161 | training accuracy: 90.38%
